# Name That LLM – Kimi-K3-Likelihood: bauen, messen, train bewerten

Baut `ntl-segrun` mit Kimi-K3-Unterstützung (llama.cpp b12247 + `ntl-kimi-k3-segrun.patch` von xerneas), misst auf 50 train-Texten Lade- und Rechenzeit aller 23 Segmente (Kimi-K3 UD-Q1_0) und bewertet danach, falls das in ein 9-h-Abgabelimit passen kann, die übrigen train-Texte (erste 160 Tokens).

Code der Stufe (`segio.py`, `features.py`, `run_pipeline.py` als Vorlage): José Padilla (estopadilla), Apache 2.0. Segmente und Patch: xerneas (Kimi K3 License, Moonshot AI). llama.cpp / ggml: MIT.

In [ ]:
MAX_TEXT_TOKENS = 160          # wie die GLM-Stufe; die Token-Scores werden gespeichert, kürzere Schnitte gehen später
SEQ_BATCH = 8
BUDGET_H = 11.0                # Kaggle-GPU-Sitzung max. 12 h
GATE_LOAD_S = 3600.0           # ein Durchlauf (alle 23 Segmente laden) darf höchstens so lange dauern
GATE_ROW_S = 0.10              # Rechenzeit je Token-Zeile (GLM-5.3: 0.023, Flash: 0.014)
CHUNK_MAX = 900                # Texte je Häppchen (jedes Häppchen liest das Modell einmal)
LLAMA_TAG = 'b12247'
PATCH_SHA = '8506eb283fd6f8c6e70d2384cf3b8ef72655d23fe49b3552ce03468d7b3427b1'
PROBE_IDS = ["e6f185340229", "428e0af3b8a6", "339ad63a2543", "906e29f13e4d", "50e58128631b", "8fcc633bb35b", "1b5e32d2b8a6", "e159d1a68cc1", "78edef61db06", "1adeeb0aa0e3", "d6c9c166ac59", "a2220d494b28", "3944a11254c3", "f115ed46c0bf", "ab056b0aede0", "4e08211beba6", "993292ea85c8", "ce1b8e9e443f", "92594e9c69a4", "e03b6bbfdd11", "a5d46f1dcdc5", "8622d9252d72", "e286a0b2af99", "92648620ccc7", "1e0c50c5492f", "e6d8ccf6b6da", "439a13806422", "470b40586614", "33952a443e91", "c6563402d8f1", "d5ef823b863c", "149d6d227d64", "7dcdf4ec725d", "870d3c498bc8", "7977f1fc8530", "500f17f53ae3", "1afb7c1b43a7", "46606fa0ef6a", "194bac934dcc", "94e45e234087", "fd98b11a8ea6", "681989e72616", "ed8aa46c6c52", "a99605a496cb", "84da4bacb5c2", "f9267f797d9f", "36a70940b322", "118e4597d170", "ecb4e9a8a9f6", "8b18a3abc4ce"]
REST_IDS = ["69b777827bf1", "b44406ec37d7", "d70a45b224f6", "0487bc7d2e28", "854d87d6cbca", "4fc0ac851175", "571dd8e6b866", "7a2381402f9e", "8418edaeb7fe", "e5861a399d53", "079b7e92792c", "07d84cd0a466", "61d107cbe080", "996a15ef95cb", "df904d7eecd8", "e84add4bc19b", "e35f82fb5be0", "e4999a7108d1", "d64322452256", "33a2f97d3c31", "2e2585d13454", "6f52a0d76f94", "d5593320a9ef", "04ae94a1852c", "68ad31a00fc8", "c805c1980aee", "34751a2d1cbc", "b4253d7cb9bd", "e599c0b5f5f1", "d08d210a6bfb", "43686820c094", "28bb7b7d3f1b", "24596d606da3", "a4e9fb519425", "61714942b7a5", "c4d63f265959", "b1b93961d275", "601fbfc33d44", "3a37f4ad7b75", "7c902d723d44", "87906f663688", "e6c1b4723482", "8751aef7d143", "4d5cd1d3f8a7", "1b50fd9e3f41", "6a8add8c18c5", "fca21b899581", "e40509f9f058", "d39ccea373c0", "b89626fe9028", "b6a8401a73ed", "a590a1f41e92", "e4eda9323e11", "269cc4dd9e39", "6ec853cb8e9f", "c63a08dc00ef", "f7469b76cde4", "030145156daf", "b7a55c6d7ba1", "2e1120857af4", "a506283976a5", "86d53eb4eac9", "c984b0818fd7", "ffcf6321f7e9", "f8fe087cdee6", "ca98973497d2", "532416986bfa", "73a4e459e8f6", "bdb5a1b99058", "042bd88f9eeb", "87e2faeac41e", "d05aa4fd20ef", "f6bd556dce26", "50d2b3137088", "917df3f951b6", "3dbc227ea0a2", "debc1a6ca567", "de432d8e5547", "a9cd5322b8f4", "33646805b3c2", "f066cae0e608", "f3374b94f1b6", "c8eb6d8a0916", "badae679f2c0", "a110fd1d5b9a", "eedf88132da9", "51fbf7574186", "6ea525c8a44e", "826af1c10680", "bff592584e89", "d6f418143bc0", "b30ec1b88dd5", "cb447cee2d38", "5eeafa1a7e81", "9bd5789acf9a", "2cc868f42c4d", "005a19083eae", "ff507658ba82", "81660f830676", "f07431cad4c0", "4824d28b82d9", "46d6b2747c0a", "cf11a1aeabe7", "1206a8856124", "1abafade9013", "6a9e5f471139", "ac6d8600c5d5", "2dcd9056a0c0", "aa9dd257b403", "ee73d9b37f1b", "6d3d498070f0", "06a4fbc2d36c", "927ce162f5be", "f5d5b8cb8cda", "daac95e3ab96", "cca22ae2390c", "138f10a017dd", "cacc4db59f48", "1f39fd737ff2", "680f81d14694", "024cee77ba63", "412157d78c13", "a74b2baea580", "5f0cc1488f82", "8734624620c3", "97713b0423d6", "599664b94f29", "51e9ec5ed962", "71b584430067", "7216d828db79", "8a0de2d16a53", "081ef1313326", "94b4862e2376", "4056b622e63a", "72135e2c2506", "5f9a92fc9e94", "5e7c35058402", "d15575c812ca", "3016a0ae52b4", "642d0098494b", "7bb098b90a5e", "6f69eb401c0e", "9b4ba718fdd8", "fae4c39ba4d7", "5eee597bd7cd", "dcc889e77361", "9ce38a117bef", "85e2fd6b254d", "263827ac7cff", "07aec3861f39", "d6b5a91043da", "66243a2591c4", "72ce0635e4e6", "9ab6c62dc12e", "d4a97afac6a9", "301de6163e1e", "415305439996", "306506de6bb3", "1e99a2d3ab81", "2c05c2ab6940", "1e1da11c206b", "1f0a576e0c7a", "10936e416e17", "5ad5a0214b61", "53c88b1f9eef", "a513d43b1755", "9424526c3f20", "94cf54bd25d3", "19343e493b91", "fafbb5a8bf4c", "801d58f1fc5a", "13054f4d6462", "2baccb5c1c88", "856f184d06c4", "17a5213ffa51", "5ef66b5b4644", "f4001865de58", "141cb12fd04e", "48207e695305", "a861afb7d1bb", "2e5d45818ca6", "608885b9d458", "2c63f287863a", "4911a843c8a7", "0041ac86ee93", "e7b9bb04e97a", "34fea1dda8ac", "021df3a2e5c6", "82629f184a18", "ca23d509d6cb", "f10090d79a73", "0880a6c597d1", "d6faf6321086", "9e6279158426", "8f558af9f872", "a73561a1d971", "1a845a8544d8", "ba69c2e54e23", "4eab232b28fa", "318e12ffdcd5", "ff50c252810c", "f3f99de4688b", "cfbce95db585", "2201b1ca8211", "b80df4959740", "c6b77404d995", "2a33b70fb55b", "c4a88dc254f4", "016063f5ba29", "2698d16bbc1f", "27a377377a18", "19597acbce87", "40223d82dd67", "98f1fe1efcf5", "3481c5274ba9", "7276634f9e34", "3dd320f7b41c", "3635ab0f7cc9", "1847cb97f8ac", "2070d5792f08", "03c0d88029e2", "cfddee32b98f", "3b460708ef33", "50552c2afd88", "c60ba8259082", "17bef92191b6", "478e27479e0e", "a1de8eba707d", "d60e3b6dc8c0", "ba9b0f2894d5", "b36820d0d459", "de550f05e24a", "2d4748c2c63a", "4a4115b73693", "6e8b5e4b3d68", "b2f0a777a027", "8cbf2280844f", "b20d2965da4e", "14a5f570910d", "5ace3a20d574", "80a2dca62fa4", "1482b33bbbd3", "68c95449f1b2", "a3e53ff6e021", "cc8cff0b7788", "552e36b6daa0", "50e0fd2b183f", "1b74fa524f71", "eede3a129e70", "97ef9c84cc93", "bf2c3940af2f", "3394aa98dc56", "55302a6ea628", "4f54b391a278", "af7f6bb28a3e", "74750b191b82", "1649aa8dd59d", "2c6be20162bd", "47139948f988", "12de70201233", "fa80fc83cf9f", "5ac50f65ad35", "6d826f0a6c54", "9f11eaed2582", "4a6a01a0fdea", "76c8ca4a843d", "895a62e6d0ea", "8c1ba8b33710", "ee0397ff224f", "f44b44ddcca4", "d628cdf33da0", "b4fed605b959", "f32a842d942b", "34ffadf2c0df", "bf1ec21eca47", "35abfafb4602", "cb67ecf221a3", "07284742949c", "bacf88238a0e", "5581ff31a861", "8c190262493f", "08c90e12677b", "655318aa6f4a", "57de7ab14ee8", "16a319758658", "d3541a4b8ed8", "4850bb070471", "10de0adfd5ec", "0e5a79097986", "07c9e7e64c4b", "caeb697aaa2a", "dfb0cb4aab0c", "2d0497e52311", "f4b956d2fb7d", "df72e4ac0b24", "df0b9b51fe12", "08eb1dd7ea88", "eb9ebd034df9", "b5033bd00a10", "32b9a795c943", "59491d0898d9", "f3e3d3d9b39c", "e782232be061", "3eb8d9587c52", "69f1a358d9d9", "ce34cda1ab7f", "e941cbc1d3e8", "d865c3d2cbde", "d3945dad92f0", "e73a1e6141ba", "f603db16b0b0", "b069f8183269", "316d8e647135", "d82fdd28be4d", "3d08627b3dac", "2458ce66421f", "6fee3079acf3", "a4ec2dc867ad", "1c8ed66b0a7e", "19260caa8550", "c14546a857e6", "077cacedfe42", "b44dada1d0cf", "9a1a50277f60", "b07625445f80", "f0fec6909507", "2cac1f1444e8", "4512bdcb758c", "e14a9d266244", "bcad1db91ac3", "619b9359acef", "c42b95c64cdc", "4eea87dfb8e4", "1ac6f78c2f9f", "d5665b99d56d", "6b4b0ea4cb0f", "7ad73cd32be6", "e67817d13c30", "c806c2359320", "5bd39a0114af", "b5f9f32e793c", "db794d4aece9", "cafcf68bef1f", "0267628c9cfd", "8f84cab7ad87", "77101f6e042d", "1380649e8a71", "1a2a29cd13bd", "1825a7307a2a", "9932c5d2d232", "0dfbaebb0e99", "da2ee37ad327", "7b48dd04ba7e", "99f05c111957", "978980ccca5c", "36b303071955", "62889d76520f", "cdd7e9248f40", "e7fb6b1b065d", "0214afd14c11", "9bcfe43fe501", "e44187bc29c5", "8c721f5b183b", "e9ac83d116f1", "6abeac694ea2", "5be24a8bac58", "81a201144c3e", "4a29d7d30d19", "c9fd72f99889", "e0082c9d52e0", "ab0a256afc04", "13b8290164dd", "82e6c047362e", "064cee6e5da2", "7af7c3073359", "4f0ebd08ed41", "0319cf941fdc", "42621fecd3e4", "f3f0e5d9b09d", "7400598f8ef5", "b6774f3363d0", "90f82ddae2ab", "b3cb654f9ec6", "7a7444958446", "f8e07d42bd71", "f7438fb11078", "c4377a0a7ed5", "1c2237e105fb", "4772aab9a6e5", "f4af707e3b05", "ea920602f2b4", "6d08a8acdbd6", "5c6ffa43a3de", "ceb5484a0e2b", "7aba9e746448", "961ff11e6d7b", "7082a2c30b22", "ccc0122edf2b", "d59c113537f4", "bd0c6a1d91ff", "e3a8578029b8", "dcebacdfaf4f", "2235f253b912", "1152986dd234", "c6ffc5192bfd", "dc288a4ea4a4", "eb5af0c0ef42", "2654b988ffaf", "203b4e5c4113", "46ef4679674e", "c2806635a123", "a1a3c1670321", "f1c5b7abc0d2", "8f280e3abbc6", "f899b03c6fba", "7ea2e5c950f8", "c9acc7a702d0", "dce8a8107bcb", "44ea099ef8e4", "f1a9932241ae", "7412c6dd2e1e", "c507d9698b18", "971ed8637c49", "25f758b824df", "e194772a6622", "a80271f8500d", "f053358c6f06", "816fb7949c70", "52164b494f96", "ed36dcd35ce6", "6e4f3d0d21cf", "1d414e46febb", "5bec887eeff9", "cc314c088006", "4098f911414b", "5977d18cba2c", "7b2ead9c226c", "b1f2ef1f45f7", "e80fcf565de4", "251e20802938", "9c94ec8d14bf", "e49571b21cec", "6f53e20a7303", "2f1e21785aa8", "79e109558561", "e2b253063103", "26d554a42172", "fb3bafba739d", "5b0512d1bc86", "4145817bc974", "2bbe9807664a", "b367cd07c802", "042f7900384b", "4a4c780ed951", "43195e03eb7b", "078a6b177a62", "793c68d9ca86", "0c38e73faa53", "2a193b86258b", "1938bc18b295", "fed44becd69f", "9a15b450b271", "2a99a9054329", "6beb26f4fedd", "6d36d0f6828a", "56b0cc54527c", "d77daab8c89e", "8bdc29951fee", "528d444b8f6a", "92f2d53dab77", "cf3402cae21b", "1c1261bca66b", "8054f8912636", "8452ba835921", "0fb6161a2061", "fc6f49645c15", "099078c9d511", "60790b8958ef", "dcd0ee22087a", "ba8fe20e8b50", "e90233b27992", "f9c69253eb96", "84d108cb0327", "cbe5b1061e56", "a3cc09b0479d", "b6c9849fd53d", "765e81808341", "4b1477f29911", "b38eccbe9eaa", "fd9d3534199b", "8bb84687814f", "06fd5515553a", "75b9189261f0", "57f0ecf4943f", "5843df5c79cb", "8c1fd77c643a", "ce921848cbbe", "3b74f25c412e", "f9ed648ea2b1", "3a5a7e30201b", "c4f429ce5209", "a21d05e00722", "6bad12555521", "fbea4a31e7a0", "f863bcbe5a29", "847f9e734466", "78fdae5c1544", "c6fba8d02cf5", "8a5eb0ad17e4", "c33f1541833f", "7f46691926e3", "2f738f830b32", "ff1b183efee9", "9650b548d3c3", "3abc8b8fa3b3", "aef7612cdc91", "ab13d66c7653", "56b2afd34f63", "a5eb39dfbd43", "2c303cfba665", "aaab859505e4", "935ce89a76b0", "dc7ea5093d04", "8f8e30aa97bb", "ec1cc7b5add6", "33625f39305c", "e3e8d6bd994a", "a868e0cc834d", "17283f864f8d", "1f990c07f0b1", "d984fa3331cd", "495c432410a1", "e04e402d511c", "92ed43afb69e", "ef76d15667e3", "de910a9574c3", "04815f740a7a", "4a329ae3941b", "838056be1437", "d1bfc8ce63d5", "fab7150f9834", "d5c69e2844f8", "3c8398aae874", "76b5e408dfca", "3539b91f3eee", "be8b87be11f9", "0eded30d7aa6", "f997d30576c3", "ea739b6a5731", "7922de5c29ca", "506cc037e03d", "1393dc7491bb", "8f8180a784e8", "69f64e03e371", "a2886e727511", "dda20d913403", "68a802b59fef", "3ef47e68cc56", "ba12c5d42e74", "1004b229ccc6", "5947945c25db", "a743f1e17a17", "a4b6982fc5c8", "14afaad736d1", "32acd5ffaceb", "2ca762dd6893", "4a974f4a365c", "ebf22557e0ee", "fdbcf0cc0cfb", "bc005b69112b", "6b3878aad443", "d510a8b15092", "7aa464f268bd", "8791e6c78aa7", "e05d0ff00ad1", "0460b1dfcd51", "4253d81cf6bc", "f24aa8942a92", "6c81bae6a62d", "d50e7627c419", "6ae6faf59e30", "815ca3ebaca4", "26ffda6df8db", "f726309fc6b4", "ca4f49868585", "c77173aafbb6", "81c62977ac5d", "a95e912614e3", "c36f5f9fa7ba", "da0c150ce8c4", "b9f17ca1caeb", "c723a858fa5e", "5dc5d3f5581d", "58988aba1406", "2feea5bac217", "994aaf258fcd", "3a26126e648e", "83f5ca9432ef", "b55b30b64920", "db5b6a2b3a3c", "c3f6fe1274fa", "5bb54db3be56", "cba8687f5e18", "094705a424ee", "70ba623da39e", "0f60bb71f69f", "1afe11b81cef", "f95cb0064353", "fd489615d393", "8c9c1d2ddfbe", "f8c152e930ef", "70ca9ff2a03b", "6a3ba150de27", "eedce5235b89", "10df2b97feea", "82f47093723d", "46813113b2ae", "c814415c3fd2", "f972644b22dc", "7d1795dd85da", "a83c9e399057", "bd0a554b681e", "1c92caa8f6c6", "bf51ce28765a", "7398d8091f11", "49da005ce8a2", "ce052868a394", "75699e148eb4", "fce9e0ff9c35", "40b01dccb938", "c6806fb98621", "391b4a5251a8", "8035604fb38d", "27e9fcaf1ac3", "7af979f083f5", "4f0d3292f860", "b4c1a2d28c75", "bbe67a09728b", "c2a9b545e9a1", "53e447917c0e", "7b187c349fa9", "c6152668a28b", "a14679d76c71", "73391dbda9c8", "1aed420734ab", "b16ec7b22d06", "3eb6c8624c17", "7c69adbb754f", "4b5b8c77066b", "eb2d928834b6", "ca606acbf6de", "b939ff60db4c", "a6ddcc2805ad", "95479852973b", "6e70aa26fd3c", "32d1a5ffffcb", "3ab8e21f0ca9", "8e5139efbf72", "847b3a8fc54d", "0d3000d2175c", "e21957ba686c", "20074fc6bb14", "f3cc4ef8d7e9", "a71457fce350", "0fa957d51253", "c3de0d2e2e45", "26cbebe69d9e", "5166ff981478", "9ffea88e798b", "088c48fc5a5d", "d28ff242827e", "91777ab721f2", "ce6e360eb08d", "ae4a8c04ae88", "be4d597da698", "f08f8653eddb", "185672d16715", "9c12485e26ba", "5b0e224facf4", "3a1109d204a7", "e9b931ba29e9", "a2f59b61a4f5", "f70213d7f8b8", "56824f0b98b3", "0d147e1b29d1", "e44b3d27b610", "bf41d4980d8a", "993b50dea4ad", "6fa5cef773c6", "5f78732a806e", "6365827f825f", "a10171505d1e", "51f630d5e11a", "34b4e79ca3b3", "2767d0ace067", "032fe4677712", "6226a37eda76", "11e5410713a9", "edd5c21db1a3", "4620c1fd7747", "763c8c58abfd", "1f608bbd33fd", "c6bc20f1d6ba", "62db366ff546", "85fd9d8987ef", "bc1c93e9366e", "1bbff0b70184", "b3cce2cbcdbf", "36ab818e8dfe", "448e67cb3cef", "f2832f55aa8c", "f35e1cb2c712", "ae88dc6bcfee", "a438338ab64c", "4f8b590584c1", "5d8e3f259ebb", "46110ccc7af3", "c14737c34b5d", "2862f848d6a9", "3cfce02c2f6a", "b3ea0328fe78", "8a4dc95be5a2", "23d5e5c12535", "b19145452f09", "2f10218c35dc", "00f73c3a3937", "2dcc03671b15", "0d9c68b5bb4b", "81c2493c2b73", "fcf958ea694f", "61f329ac925b", "8f39914a6d3a", "d9f053e5c7da", "a979b83050b1", "873bc8509da7", "1c7df7df7d5a", "3ba6e39a033b", "11c4d387cf45", "a90f88f19639", "c00c684cd272", "cd6a62bcf77d", "8fc583b26346", "5d54182583dd", "2c10b3024c5c", "1a09c67d51f6", "e1cbbd94eac1", "8645ed8afa9c", "fd3b972557d5", "8d0369f5ec76", "fc9a710647c8", "79167e97eb31", "784b59d2f503", "b58f9723ee6d", "c4f37e33dcc9", "ca49dc0856ca", "697cec6c7737", "0c9e6f24f550", "31bca31ba8fe", "330022ffe8c7", "b3afce225db9", "b4b26abd7493", "04a9c809ad84", "6166c606c17f", "90ebf362aa8b", "f038103f87ac", "64932ab97f61", "3b2a05a616fd", "832773f8b2cb", "30a643c9887b", "24d20a655ef7", "84cc220983df", "63f92065989b", "adf269acac5d", "51b2c7c31cbe", "954bb19c9f9f", "80b5cbc6511e", "a19d42e32196", "9a316ed9c2a0", "0324d70bc185", "fbce4394d535", "c6e1ed4ca9f9", "a2460f68e686", "77d20376a0f7", "69c13bb72b54", "b2e8a5f19441", "a41dc9beb51b", "6d4bc94f316a", "70bca41dfdc8", "3eff414954e0", "c2a73b34f308", "9c239461481c", "016f17711735", "bb6e40cbf7d6", "3e07834f0360", "ee3196872276", "53e90eeecb4f", "43f337388fb4", "4e3656e30a6f", "a758113dbdf7", "3a0a1be872cc", "03a7d199d137", "7479af38ad07", "415a5c18d9f1", "da516fc381a9", "625d2fa76f12", "f3cb0e84a90e", "b181f74a4ffe", "6add70619ea1", "e55a14826420", "5a9411dffff0", "228be2c500ab", "9854e7f6eee6", "c2a17db8a673", "bb4918792866", "b7d943015687", "20a40a0a16d9", "1dfcf2f62bc3", "867584e134a6", "cdbe9b4baa33", "d0c04a655001", "fde07ec97c18", "367ac362d0e7", "de9587bebee2", "57ed03e23b77", "f82d694c7819", "7fd8c99ebc00", "e535ec1c990d", "885852519745", "abfc87bd4183", "a3fbc135c19f", "1e24e46b4ac5", "16364cb47b35", "080e9591c516", "fc0a9d0b85e1", "a021878baba7", "8312c8912d16", "164f17cde066", "4c4235c98f21", "2fcd94565092", "031b781f9e30", "2a9fc2029a59", "1c38e9ab5670", "39296bfe1794", "d252a691a8b0", "40f9b5d8de27", "764a83d198ec", "b71a329155ce", "414ab924ec05", "e5a56c5429c7", "bb50aea93542", "9b1e4b0f7c84", "76d2692cacca", "4a203cbf2d2c", "f7a3cc682147", "27ae94a65382", "ab10e3df96ac", "3ec973c4a3d2", "c989b2026ed1", "3d0438688cc1", "a2dbe4512ed1", "d4d63ea9f858", "2975f5ce4d23", "21fa0a2083d5", "49a0729c3515", "b271a81ec0a5", "fbcffcbc4280", "28f0bc945c46", "c4d8abb2a4e1", "120b51d3b72b", "9af6e78eb9a8", "2fd9293455fe", "425c0e27f253", "6a20f7f19b11", "4afcb442a0e2", "5f7e5adb2b35", "b4f070613787", "7516fe0d0e9d", "90c7ae725776", "d5e29e0aca82", "81349ab82c8a", "a087861a6423", "ebe50a83c734", "accc5be8d921", "f18e9e2dfb17", "7f2308d2e6ed", "f2caacecc495", "f7fa6c0182b8", "75e04f2df672", "d57aaaaa9b0e", "62cb74ffedd1", "35d5ba1ced70", "fb6a3d3adf96", "6c6767295f85", "85b62600616d", "e4587b924a37", "9eecaf60e8b7", "8a8f16627c4f", "ae2476d730c4", "e78f7f1f29ef", "1ea1e928eabe", "a36fc1503fd4", "7d56ba26d184", "21002632ba27", "92df206d66c0", "32a37f767f83", "dd1287b26649", "3b63fb5bf119", "f39d5f7acd20", "c928fbd16c8c", "f5e35bd59ead", "db874dae3056", "7d04084d5c69", "8097944012ec", "aec462c207d2", "0ea416e2251a", "34767983637c", "d5ca413f8b2b", "05d4e06a1dbe", "91c870f56286", "20dcf6a9cf7e", "9a9d425b6ccf", "edb7bf040522", "19732d759ba8", "f2e7357a2c3a", "72ef54139e9f", "7faa6e0fda8d", "91a189ef0a4c", "11770123c25d", "8835cf8a2ec6", "85173bae963f", "3b524d6c9e53", "2cce99aa097e", "cd0b5a60a151", "c662f8ad3fbd", "3ae1aaefce46", "59f4a10e97da", "2c4d9a2bd769", "4ba98db85cce", "79c20689c9d7", "9f98b7686046", "0e7003f003f1", "e975090533ec", "08302d2966c2", "5c03aeb78988", "b701f313f6a6", "3228ce12ccff", "29b9c4707f35", "2b47b6fe2cf4", "8c6f46e6cae9", "d4c73c0607dc", "da6a4d706669", "8edeb69fdc0b", "ac01bf3c34a9", "6c481f795562", "72b6185d30a9", "94e5ae808ade", "b415634c4cdf", "b194a401b3c6", "0151ce9f9c9c", "4f41f893b6ca", "3109eafaad4c", "1da864cdad85", "1bc9af4cfa7b", "361cc72ec05b", "4c2cec8f2c3d", "b8460ad3ffcc", "0f4895ef09e0", "7f5001d315b0", "e21d7a90eb0b", "47e2db211cca", "462d342257e2", "e6fcdab1a4a4", "d84c838e875e", "e1bfb786d3ea", "dd9dfe4fb4fe", "3f57e2373523", "1c2f75c53b3e", "6198ca420035", "a2ca8b10e94d", "9155ae9af567", "be9b5cb6bdfe", "e234eecef3b0", "52ecb1764e5f", "cd45dd8ff4a2", "9153282d16a7", "ca84830040bb", "51c8b92674ed", "86dce177b51a", "667787e71542", "8b5f109da8b0", "b8466eeee10e", "96b2f8a0bef8", "6140f4a12ff4", "216fcb0c3124", "4cf9025ec69e", "e7a918b3c22b", "80bca008e03e", "fcadf900e6c2", "7449b7a7b02f", "9defe17c28ac", "3f81b06fe8a5", "5747bcc6404e", "684999e55269", "38819a8d895c", "2407cd15b024", "43895520ce86", "a9d8fe40cd58", "02149f5e4a5a", "774f775a8bf0", "8ec539e9d5db", "23433b17ee2e", "125a9f17fa51", "3f267d731532", "fdc9299cb94a", "38bddf0035c4", "100a44b50502", "c4dec47c3177", "5445f9438edf", "8fde4dac4ef8", "1e99ad026d23", "06c2eba55e4d", "3010b4f92995", "b30de33d5e4f", "93bcece17d87", "0b3b2d1e6605", "53be8c458906", "d37090618f6b", "e801386c46db", "c3ddf4f08536", "a935c1dfde81", "2d99964a1503", "fc855fb03952", "8dee43fdd1e5", "944927494587", "bb17439a63a1", "46409161556e", "365f33009e27", "5fbc67674104", "ec15f2cdaa59", "19a1b858b379", "a6c8757eb26f", "7b07b640254c", "65fbfc3875ee", "846e8d24de9e", "ae502a011510", "8b0d7c46fdda", "6cef7df4df76", "31f2a9115786", "b697a58e0c04", "4414b2cd1fde", "3637b0b3227d", "e0f587f8abea", "f25432901e2e", "ecbdee773f9c", "c1478e6e7518", "79e501c0b519", "cdeebe147f1f", "3f6041a179b8", "40a160b96b03", "eb3ab20732d5", "b195674187d0", "ebded657bc25", "d10a7051815b", "9cbc5abbc460", "f9be3fe7fe12", "30363ba24cf5", "a4249dced221", "cbbeed87e86f", "26af2c0d895b", "8317655d2096", "e8d2504afc5c", "f31362988dc2", "eebcfb7b8814", "5cfbcf9f9794", "4a81e1b2cd48", "c576ac2e8381", "f81b7af61908", "d478cc943571", "3cbd770e39ba", "58bf99c05371", "168d8f48b33f", "44f0e29b2f70", "e5a62cf03846", "fbed24ff3e46", "6bc7c3a4ead5", "5876b3ea6a76", "81ea246a252b", "32efb2f508a1", "7695118c0952", "63668ec1081f", "6fc51978f9e5", "943a5a365020", "0547d817f90d", "3b4934fac059", "324cfd1e4183", "52885bea3def", "a05290dd6911", "0b96a6076329", "9a06e3d9f691", "44d4d6e504de", "da11916953ff", "4ae6c6f97b58", "215c3432707d", "b2c8b43d2734", "29fa73608693", "f8be469c9a00", "03293c742476", "f9aa3bf0cfd2", "19b5f954f562", "79c7e8341fa0", "1cfe4799cec4", "1d8c23e637e8", "e82c74dc9001", "1a2060f80a6b", "54f6b5a52fc0", "50613856df91", "5e3344e590e0", "690c8c8447ff", "a15ff84c3735", "90c1ebca3270", "961b1d8cb766", "1d1f795b41f8", "c3c7286c7f71", "a485038b0f56", "5f26133f4764", "74c6724feb00", "8c43615fb687", "396a5e41c431", "a481188fc2c3", "bd438f7c5006", "a9fbc364941b", "8139cfa4ea13", "8d6fdd5ceb85", "b8cf29c286b3", "8a4f6296b27a", "c14a2add12e2", "b8ffa415e2ca", "991481aeba27", "50af1e88123c", "896d36b498ea", "95d18d97e74c", "9f7b21ff7f6a", "9f0525f998c9", "2efea1a1ac95", "db3f4a04db54", "5c70f840059b", "ea15fc28ddd2", "6b755712d0e1", "6df80a073449", "5f39545ce202", "e123d91d5452", "225d0d7dd4f1", "a7fd04258d69", "e1dde2a6d3bf", "4dcf3f316bda", "5d668a62821c", "cc377720d306", "5498af326141", "dc8a41e465d5", "05e69ff66166", "f3157aaad444", "6db8bf77f15f", "587008d1a0b8", "5dd6dad55ea8", "e2f129688f03", "c486f25c06b0", "81cb2dec61b6", "a2c3d46b07c6", "9bd6816b92f1", "b17660f3f40d", "f595d3bc2328", "8c25456d0a4c", "9ba9094d4929", "8e6fae8979c8", "85f9055dfe20", "1cf78f8a4de4", "3548c5a2ae5b", "06606903cea3", "ba0e48ef6a10", "8cd9b1d040a0", "d413f9b9b139", "6b57957ca863", "b494432b2112", "0c2f3b541165", "14aa877aa676", "80597873671a", "b0ede42bb45a", "807778979797", "673c7ca86ff2", "bd7696450be6", "312c02b7e128", "8e4b9203aeb1", "518e41acecd4", "b90fc0b9e6af", "83a95d3fa0bb", "664dd5a988b6", "69800d19fc04", "7e78f4a7955c", "8dbb88481167", "1528d9c4a631", "8185c9454316", "8ebde7660832", "298a7388016e", "9a37097e6a99", "1d7aaa83e17a", "b3f6147ba5bf", "05fd45356e2b", "22a840fd5885", "2029b385e7ad", "35d74a866e45", "13e6d5b5714d", "f74775dddfee", "c20d437aff8a", "805e41265305", "2565949f4c59", "5f7c7ed0efb8", "46da3aa3c840", "75edb7864957", "c0d00726bbc0", "76914d0ec2df", "f5d50b88267c", "7d16c3233822", "fb5f00b8a5d8", "48d1280b660c", "a3b1ceb31e6b", "13969c2b1824", "c77a5f306917", "3a6987de1805", "3a9cba11cdf2", "107075abcd41", "17869cd692ee", "4143d9938924", "d1ce3c46631b", "4d7baf004f10", "18aac2816e9e", "b043e3e8178c", "fd6afe9f9dae", "560f3a07e54c", "10734f1aaaa7", "7a04905606d3", "f948704c62a2", "0d0904188f8e", "c570e323ea2c", "8322a0fbd795", "802002b4e438", "835c35c6050f", "41eacd26fd1e", "4c6e633021c2", "855865713143", "6009bb423aec", "60bd0a3bf3f2", "2f3f544ec499", "3bf3de4e8f9c", "2fd365d672c4", "90a76e5b3610", "dc0119372b91", "5cc9d1157b59", "eac2e88b3662", "0fa4ba4e39c6", "fb385cdac722", "56255ef5bb3e", "472fcb579877", "53eabc6499db", "bafe704c3beb", "f6ea13c21c0f", "87260a8ef8ed", "84cc1c0604b3", "75061050ca07", "a49b6ffd05e5", "f55ac93f7f1c", "a31835ac1877", "0ffa23329f30", "d8d8b7ec2362", "86f0e6483e3e", "5e11b62bc56f", "180ef269cbd5", "0fd2fdf3bc1d", "72b1de21f21a", "aa9b2904af1c", "068c172076b8", "6bdb19699b9e", "be7776510a9a", "ee52bf246852", "925cc7b663a2", "a877f155b50d", "ec8229c85bc9", "5eb3dbbd8315", "0f5fa62977aa", "035cb4ff2861", "da012bbba825", "8a0530b06246", "9fab1b72db28", "251878a4a004", "0b0fc6101737", "ec72c61a2a16", "132924eaf728", "e54ad89e8bb0", "6be7cfdcb0ed", "6b77c9f9a96d", "1d78d8f996e5", "0c5d4398a954", "172010a98047", "9f4d6e04ea4f", "ca136fce936f", "f5dc8e32289a", "c73437324eda", "121828f675c2", "f8ed78aa6b0b", "2ba628e7ee2c", "2c318c759180", "7f8e4e900a6d", "5c38e1d2cc95", "0b3139e017df", "83a6b473f73d", "c50b8ff71314", "f4f05bc6723a", "2df1c8cca6c0", "7692aa9ae7cf", "c5439b638fd3", "01e81e2b0197", "08c19ca83752", "72e6ec80b181", "d1b64435529f", "bd9686531247", "ec8352804a0d", "6b15f7c40bef", "b78f00df92bf", "6be69d83d8e3", "acc988db4a97", "71873700a6e0", "aeda372d15ab", "8cd58965e573", "87336936142e", "86a30fd1fa53", "06a94bde4ad3", "1a6913ea9402", "5d44e7c14bb0", "6343c31549b0", "65cadb43a376", "3508e793fad7", "a1660f8fdbf1", "bc6ccf5306e8", "e74c9aad7c5d", "645d749683fb", "9d3cbd169c62", "44b783cdd0ad", "4a9b75c4fa18", "63aa55592200", "2bb124419619", "484075078abf", "1083cf292e00", "92efeac7a860", "ce13550164bd", "c7242cf47316", "44ea9ec1041d", "eca8954e07d7", "5c541e425fc6", "be74a0c3b4eb", "3c9f5702253f", "5f3019fcc38f", "e3546e666991", "7ed463c8037b", "246837e9380d", "c474abab7004", "e8478e57adee", "7d565d5ee375", "163aaa89597e", "808ba165b2dd", "bedf912992a5", "a9fdd04db099", "24bbe5247459", "1c37baa6f925", "91d976f99c82", "ea833758cb75", "703d0fc69418", "b6b3cc874fb2", "33f212f2ee7c", "f5f8c20330bf", "9a870d8d4c91", "b1448dbe0fa5", "61611f1aa5a7", "21ef429c2448", "fde3bd75d150", "520b3e8e41c7", "0408e914e870", "97691bf86697", "02b9b5daabe6", "1ce5ab359716", "625d6ea32ef2", "f828460ac473", "2d139073b166", "48d6e421432d", "555468883c2b", "929a8ff4aea3", "2227a7a42f79", "aeafef3acfb5", "ead512ab8605", "fba4db3292c2", "dee86756b590", "f4f9533c9292", "4a0712cb9a18", "4cc61e1b9f1d", "c5e492ded59c", "88b98da527ee", "f1999f03a6db", "acf67b48329e", "290fedf3204b", "7c095756aab7", "96aebe84a2df", "eb88075416f7", "081237447f0c", "fb3bf8c02bdb", "a5a2d27a52fb", "c086a718a031", "51369f187274", "b323aaece2f7", "46d022892100", "56041b4a75bd", "aefcd0a0fedf", "6155772251b5", "a3599bf82b2c", "25b996aeaadf", "dc458d7e5c95", "f1e87992700f", "52d11edd2bee", "440b573d8c65", "f299f0eceaa9", "3fa10c4408c4", "46dfd8a8d62c", "191a9c01dd35", "38e6567faf17", "8280378024f1", "534e76f7e5fb", "b745b0206dad", "cc1d8e6f0f67", "ddc65765ec98", "c68b6c3f4c6a", "97b870807d44", "fabe1f706a06", "2fc5ccb55133", "31507dff58c1", "b9d7ed92bd7e", "e1f420dd034b", "8f16eb8ed923", "694068cdbf6c", "dde149755d90", "bb4139755803", "49e280519c50", "f627f69a741d", "dade4181e8b8", "d2eb56338ba1", "332f60477857", "e53dd5aaf5c9", "2c90bb91f33e", "ad369bafb1ac", "9422b76ee450", "1ee3ccd66ea0", "4fa1c807f0dc", "adc8a5f1bf68", "4b28df4a91b5", "0d1a676bdffb", "299c21f1afca", "59d279848deb", "317d08569390", "a7b2150e045d", "f606d483e8b1", "d778e6f0abb4", "370c6ded5328", "fc2dfa9082ee", "59a13bce3f9d", "35991ba25829", "8b1d940ef769", "3fb166fcad94", "283d3eb91d6c", "e263781a3239", "1272a3b79a4e", "dbdfc36b289f", "4f1413404435", "d74f1f11d36a", "75ef3b9a9331", "d7d6da6ce0e7", "5f4f1e4872ab", "9b5e8fe6f4dd", "ec5b8bca91b4", "9b985531f7a2", "702f4268f2a5", "2959fb760e40", "cf04fc80233d", "94415ce47ec0", "34168140489f", "0b6e0221ce68", "dbf66e95f059", "ce64e26cf65f", "364c151824f9", "b266b4a388d2", "156e486b986b", "d1be00c5c707", "0b4d1dd59809", "46b9cedf4627", "475c1f77b4b2", "bf921cdfba16", "5c017d46a303", "4045787bdf8d", "6f1abc30dec6", "cc5ffb076875", "7ecb28657976", "49447711281b", "8b819384aeb6", "2397232a6c8d", "2b18cc4cbb8f", "24d36ec3348c", "1c4c3422b43c", "4ff7bd8f685c", "9714de896a35", "d8ffc7f3ab42", "e1b79d6a05fe", "6454799d231e", "a794bdb37295", "e120ff3366b5", "fcc203926a20", "496fbdb6742d", "4776b442ffd3", "a45cd364068f", "4e379578324e", "1b8280851fcf", "25cab505d1bf", "454fc2e4a4f1", "c550f2f970bb", "eb0e157534ee", "3906edac71de", "458803bad50e", "525fd94f7b30", "c7c0e8828a0d", "6b528406063d", "46e6180e4dcc", "fddcdf032ca2", "c2cdbc24ed67", "fdce0232cf8f", "cb4d53fd1198", "c3631020fb8e", "e6ed34a27e93", "8e66cdd2ef82", "f1a6f8a80dcc", "7e4dedf0a3f6", "0d13cab9b773", "197c88454139", "ff015bd1745b", "64f7774262a7", "928785b2080a", "35ee45ac4d29", "6078a4010cc9", "a2969abf3b9d", "c91fafed7276", "7f90a37b3be6", "57869dc05c91", "b96ac2bc63f6", "9b4d60740c7f", "e9c0a461e3d3", "a937d84a530d", "571a84845f77", "267fb10831e6", "7f57d42d5379", "cfccd4971eaa", "8b339a1999d7", "d390114d1710", "f639a9d23267", "bfa82ed05047", "07497c674f0f", "ad6b55a045d0", "07a15b7b4560", "9d021c4b9cbd", "e37d41160b64", "6a6c1201e18e", "522e0803b053", "07bdcbaafbac", "8e777f70e5b4", "e9e15dbd7821", "6d9d030ce35c", "290de307b6ec", "56e627ca6b7e", "4e85bab2d711", "bec61058c67d", "e2287e84637f", "df1593e19c6d", "665ae6f1a9d0", "bc2a552048ee", "bf536b50806c", "1d0ded3e3308", "c37d7210e253", "e34c65f4bd82", "6843521dd4d0", "5539ecd38f02", "f9db21f97f38", "0c620285bef2", "93f24e213b5a", "bcf6ef4aec60", "d1d2e94d202d", "85742dc3220b", "4e15ad577255", "d7dc8f09557e", "d5e2ce4f0ba4", "90555da599e5", "a1d544dd030b", "ec13c5b48977", "97d9b0495730", "93aa02f1c0fb", "75f00cfd8b0b", "70e4ed001991", "12b08650d0d7", "1dfb0d7ccc4a", "07908fa90ef0", "3aa6a6f37d00", "1e4b027a5ff8", "74e26d67c9d9", "f39ebfe4f0c8", "954916ddfa00", "bbe152bcea15", "4635ceebbda5", "e5fbf112efc2", "e7580b2c8558", "e11456415199", "8aad51e910ed", "c6ed855bfc9a", "0fd374eee12b", "c4df1cae52bf", "bb4641543e4a", "85d318bddc0e", "6a41276d478a", "7899be155b33", "bcc5944f94fe", "bdbcf2f48026", "13a0686005b4", "70f5dc70ce96", "b3df2706fd46", "31d984e087bb", "b4779ff8fb81", "46ea6a8e7af2", "8c5d506ad4e5", "1b587ecedcfc", "e05887d7fe5f", "be0c90cfe7e1", "8805a4c58bd6", "fb07ed3f0ede", "f3bd0689be18", "3ac6efb5df62", "f05ee4898654", "7fc5331805cc", "c5fb1299c8b5", "32b2bda0141e", "29ece51cdac6", "df3804a2b27c", "44844f30add2", "e16c3e4632cd", "0eecf3767d73", "b9b6bbb0c03f", "b39cb693e683", "34437cec3b7d", "70e6612eb00e", "9003cd62bfe4", "8589a1dfa216", "9f85ca42c010", "c80da41f49ef", "84c732ea3a52", "74b6578fc690", "3c795456d75b", "15b1d592d26d", "2b0cbe7653d2", "89344f253f29", "3bf62efcdb85", "cf1e255f2472", "c7737e13ac3b", "aa2337a3fd7f", "a5f99d5a99dc", "ad55dd0e101f", "37aff98914c9", "b636819d52db", "239872914e72", "58d93e7ae78d", "54c1cbf03982", "1d9f8b7cb49b", "bf67b863bffb", "fa45822c598d", "7faf6c0ff285", "a4084e6740f6", "4a2caf078aa3", "a5c2c2378de6", "01a2491c3b57", "11839b25436c", "49297e30114b", "c47dd80e0344", "0f1b9785febd", "7ffd0eda2146", "893fe58dd99f", "357238fd910c", "29a16c36badc", "4780a1d555b3", "736c6a962c77", "a966824e4534", "4899fe170e2c", "7bba9f6c8a56", "010967f56aaf", "1f317e09e595", "340c8339755f", "ef830a109a0a", "da3aff01a641", "50cc76f8e4ae", "54fff95e4a09", "8b9a6f89981c", "d7270f420b67", "0a1f8c262e2d", "04ae4c51f0ac", "0a9fedaac247", "8ef5d31e05ae", "86f5df290e0e", "0b228a1654e1", "29ed9c4f2312", "60239a6ae8da", "17e99c54eb61", "f107e806d0bd", "dbe0b60d01f9", "34082b6a2ff0", "b85337078336", "7dddc66b8957", "92733ed45778", "055ea7c1f861", "4b49d97ec4f1", "531f26620385", "6f9d2173ce7f", "ca0f3d2cf5e4", "2cfc961d4ca5", "f167a404c88e", "9b46cfe19257", "857e985c29d6", "d1998fe0fc5f", "ecb5d49ebd96", "c42d9df62c38", "5dc4d88c66dd", "fde38289c3ce", "6545c9754511", "16b9932865c9", "dbdb1e07bf5a", "38550e47739f", "c5cf965c0bd3", "4031b9f960ea", "226bac377e03", "a86e61db8e09", "87efb24fcc0c", "3eec64da4bd6", "dc7534c4d8a0", "e9326dc81a5f", "91ce68c1c2f3", "1e7c4b294ae6", "0433628fedae", "413bbbbab55c", "017623d68134", "df070b961e03", "df267be8113f", "54d844e14c00", "2482f6341591", "70617dae3b30", "022da1d10419", "b16870f74ac5", "ab6d7c0ff501", "9dd5c95b4d7b", "0f8c1f908643", "074bbaf9db24", "dce370315c55", "6487229de6ba", "cb2dc88cf1db", "26a5c24b23e8", "8bad1e37da21", "e84f466518ff", "954399dbedfa", "f62eec07d82c", "6e361f1f9b4d", "d77d233e2c90", "0b59b36ca0db", "a14d7e1499bc", "744988a64675", "7369766be464", "acdae2b5615c", "3cdf18ddf73d", "fa1dc2617e96", "4c1bd97fdad7", "ee8a2d661d7f", "d254e390713f", "1b25b08dde25", "ea87df425a56", "ec402b924caf", "2771107f015e", "1a6ecc8f3728", "a2f0d56a297f", "950981db298a", "71c1978bc064", "0d0238d265ae", "be566b8f1e35", "caee23ced866", "055db27df510", "205e82028d70", "ee0dc7b31eac", "adb7d4b8a5c4", "650fcd5b3936", "2c0fe09d3575", "490f99bea577", "1d50e7db2a07", "fe7fdf22bc82", "30b3e185f03c", "4eb988e8a0b2", "4f4bd14f3709", "3809ac7f6ab5", "efe447f6221d", "684a6eda529b", "fa5263d7784f", "5c03576b2716", "ef69f5a64ab3", "6a7b1884b5e6", "546c0965158c", "1375a41c25e2", "9ca02a2c58f5", "76315b97051d", "c9c75c3566d5", "4773a5977f47", "154861f20a91", "e9211dd63fe7", "e4cf550738bc", "e8ebc81f105c", "3f26c7a25849", "561a3c422856", "53a3f0f280e2", "b6dbe217bbb1", "df08814c63ca", "cb087de3af67", "8050b7d9f39e", "b26ec4c59146", "f8cab482ed2b", "39b91c618fdf", "b9047e18d6e2", "24bf0654eb1d", "478c3b1c15b5", "7d058bade110", "d47a85d4c54f", "ce9908dc476d", "ccec9e90bcbe", "93be3dffdfa5", "3c4a89eb9de2", "2ef1c18c23bc", "e6834c3cd6db", "5327dd850e2d", "282c8c47afb7", "927db55dc6d1", "b45cbfd39bfe", "d0d2a70a7664", "b1917b364b1d", "7bd71aceaf56", "ce0083c48dec", "80017f709827", "b4d1bfbc82f3", "c7150bc9cdbf", "71aa0555e6a8", "85bbf22c085c", "5114fcac6158", "c04b34255c4b", "e2bb5d3ec823", "0994248d255d", "3e70864eb8c4", "7b62eb86245a", "6b7a325650e9", "d1ce4f86bd54", "509dbec4f381", "6d3da13ff9e2", "218937517907", "595dbc1b0fb9", "121823684cf8", "d7d203bee40b", "f48ffe25d2d2", "6391f13e18f2", "55c77738e561", "b9c41e5a146d", "26394b5b848c", "19d687a0b0fd", "ccdeb6881565", "da41953a6519", "cba6007babfe", "78a7f63d0d21", "bb93e9ce7940", "edf8d8c0076b", "1d26ed1a9a92", "9303b03ad2ee", "f3a4f8c24466", "38bc9f95907a", "d9a234ef935f", "0b7939380dd5", "2e99b3879253", "43d4e6ccaffa", "fc5796e84f88", "8078d88b7ab4", "9b5f44324da5", "078d2082c94c", "6af46413626f", "1e1a9474c044", "0831bd701808", "6c684ac28a32", "5f2d19843724", "5a7e04a02432", "e00de085346a", "78372cd3e2a3", "e61e8ea9f601", "a423888802fb", "3f445588a662", "f0476f627a5b", "079e1f52d91d", "55cf73815bba", "a7653b15b019", "6ca532e16c12", "49a12a2c73aa", "857db7e94cd3", "605e39b01a0b", "1d3f50391ed3", "00d3e8c92596", "f05fd86bb630", "a179004ab0b8", "0fa6c2c8eec7", "8a72d2302475", "e8b514dd0e36", "929c7a9c7ace", "2e5bd3466ebb", "050837aef9a3", "dfbb8ace6444", "0663836c7892", "b495938b635f", "b17010b47bb6", "86f03f1040b9", "fda84daa4bf2", "ed44623869f0", "cec7e5d3866c", "92295a4896d8", "e9ff3857c324", "b910fd48bd9f", "d63996430ed0", "80806f71b61b", "fb26404a0468", "7a90af954db5", "7a65e9521b02", "b102ccb9c210", "fb8eeb053e63", "14f02c878633", "ab5a9272cfe0", "1d56d68d3607", "c41a25aff6c0", "376b6c1bde30", "a3bcacceddb1", "772f15243b18", "88ac56dd7111", "51ef0915f555", "c7c1ff6c0407", "9fe582ace63e", "b159e6d01158", "6e995d1b93bf", "8a1eee84bfa5", "a118204e3101", "5ee8de14f1c0", "b9ae5d39c90d", "567778afc902", "891466a40217", "85f5b4069e21", "530ba671efc3", "53cac0452849", "928526085040", "fa8a9e35117f", "82e983417320", "166ca40c8eec", "1468a6ca2987", "7109662fd31e", "2bfb69367fa3", "44633a9c8403", "3705dd46fb93", "4417521312fe", "c302322a1460", "7423c63c95c5", "3d7e54e94a54", "c3e4dcc080d3", "4c852b180047", "428e494d28e0", "7f8774f2b295", "252874901bdf", "efc552d84ffe", "308927d2cd4e", "0db208dbdab3", "70ddb23cccfd", "edcc330cfe67", "032c15c0abec", "c1bd1c398500", "2ced779b3235", "69f88becbe94", "f3442b183256", "7e422df04c48", "ce43814e729b", "6d2953018970", "66cc4fbc709f", "8754cabd09a8", "57145890d6d2", "1e7d81b11c47", "62bcf9f36522", "5acc5f2b5425", "e18e6a68df98", "e45c91b6cba0", "941129c17bb2", "20098091727f", "f15e47914a79", "97edeaec596c", "b178ed2da44a", "cc9ff2cbafc8", "5b7cd67fcaf6", "be8ee2ca64d3", "26cdc7c4a2d0", "a7ca2d663a3a", "566a653936f4", "8256997fd45e", "295512dcbf29", "e74ebfef888a", "f015eaa81579", "6ba3a5e959b4", "efe83cd24226", "fa9c30a28959", "15d458d13d81", "5ae5e0352d46", "41af50534d1b", "c58db03d11c7", "4b3a6cb02633", "59f38f29e4af", "72508e1c12bc", "66a0a0fbb6ff", "a67db57a0454", "df886c82a096", "66d04908b842", "b8cb9b4ebd6f", "68cc30247de1", "b9d2aeae0053", "dfcb8d4a740a", "83e241a8c9b2", "a004d4d7f156", "d9a7991e2d15", "c47f3c3c4ae6", "afebdd254e3a", "894471fc498d", "22b5b7b55b12", "8c6656f07e70", "0fa86455a28c", "98ea5cffbce0", "d8f672a811a1", "a0741dc73b06", "7b67e8aef03a", "656967e474f4", "c5aea29bedef", "fb6fc5634d10", "9fbc0bdd15f1", "81eae09d95bd", "74bc07cf222b", "3e2364945221", "1d46d1ccfbef", "540565c3be5b", "5e386c13ad03", "42318c47a990", "495e17053625", "838c38498740", "1249f0d13849", "1270a25e1d7b", "69d8100f83a0", "e63bcbb6e37c", "ce2ed538111e", "cee75b5b422c", "5458cffec563", "d55bdeb9cbc7", "fef0d2a074ea", "fe96750d2f40", "e8f2dd638713", "0c244cdd8949", "2afa70645b39", "51110ec0a215", "90bdb8e7f078", "e55a0c04316d", "5038a19ae113", "45c98d6c9bcf", "491d904bd183", "a32a3ba012b2", "0e585a033c07", "2fa0bc220cd9", "ea10a9bf1690", "9dc00b6ab3fd", "b1e63aea1b60", "c947fd2d06d4", "3d3398b0bd39", "e59b9d1c1782", "255716779db2", "1d8bd20b93dd", "c0b31d750a32", "c5e5e26c8b25", "4e2c924764ae", "933188cd9771", "e38c5159f57d", "44ccba12e7b7", "f2ccf03ec33a", "0ec4c0667cca", "c94b56673fd7", "59f374c029ac", "64f622da10c6", "b5b93079c960", "b70bfd05c8bf", "97887dd87795", "e71b22126ce1", "16d413ad1fb0", "99af1c48783f", "7519d6d555ae", "1d14a92a02ce", "596b609a6506", "f77b29258760", "2dc574b09e59", "1eec4f39ae56", "07f409f75b21", "46e332753af4", "014fcc7128fb", "f2c97b214ede", "dfd30d6a1d34", "6b8965f217e8", "75e09e2156e7", "7a59fb4ae870", "b8a9f94a3086", "b41a0c6ed641", "10b9982ec8e5", "de15dc1498a1", "15a7a9ff8d72", "b79cfd41eacd", "33daaa0c9669", "e3ac4e76d5a3", "3921ed4a3a1f", "e5ee44debd39", "ebb92a139ff7", "b3361e46bb67", "f95eb3f115f7", "ed81453f3c87", "2b196255f6eb", "5eeb8a21c882", "cacc150b7c8a", "8f7ff704121f", "6fe0f6df3704", "14f5c74682d2", "53d13f829c09", "f2a8a13e4253", "604283101755", "0fbd9e64950e", "cf2f4626d0ed", "3714e33d3f77", "0959253d546b", "352c7500b47b", "9b84928f46f1", "fc7e43489460", "a02a3048d4d1", "e7a6dd3bdce0", "3bac62ad3fc4", "550a841df116", "f5b9416e28c9", "9c045c644d4c", "793349c7e4b0", "b172d4db3dbe", "b873d71ef034", "f7783568dd99", "fca2dfd70314", "0c93c86d3028", "ce88be0375e4", "145abfe0654d", "4f967d029538", "4eb8265a8d9f", "25e8142fc679", "adaf1d14b287", "93d0a53d6ecb", "d3672fee23ee", "6c28626898fa", "f89858c58633", "72ab2dd56bb3", "bcfd99a418c5", "aa1ca4669e93", "8d6f307c17a9", "d28693ddb64b", "b4c9cc954948", "4e1df414e4b4", "e5a4a230b0cd", "9a676a19c8cc", "f21e4a81ecbf", "d4be4d33d4a3", "4e77cc910f1b", "71f856fea55d", "d59307d61c2c", "b58f6cdd0114", "642087509d45", "678429f81a18", "d583ddf63b10", "5f7ad252af07", "da0e3e687358", "2db243903ac6", "983bc9c4a262", "f384902d324d", "34d4137b7380", "f6a03d6833d5", "59420f005127", "17a6485bae0d", "053b7246612f", "f97daadafe5d", "81cf95acf213", "5738745cb5c7", "2b9a4f6d78b4", "35c3137e866f", "27e916be5018", "6920b9f2c359", "19cc29f7755e", "7ed5d6538654", "626e47bcb72e", "04e801ece64d", "e57f770a2853", "98457491402f", "f87fbd5452e6", "8e377b8de401", "3ec79a236444", "ed73482521cc", "8d15e909fffb", "3164365b3bfc", "af13885625b4", "652bb0c1942b", "eabb7436c119", "6115b51f102a", "ae1b343e869c", "43d45aaa5472", "267a04d60514", "41b482949d16", "41c4f181e4bf", "3b30a59f20d4", "c96cf7c72d95", "d52bc7b9ac5d", "61ee84c999c5", "b61cef4831d0", "77224ff1a380", "4a839bd771af", "2b5d7757d9e4", "4ce0a52d8b2c", "a011dff6d7c3", "c62ec4ae4462", "79788ce85e2a", "5aee813f958d", "e3916dfd0211", "7e7a4d109c19", "bda0a097a2c3", "d5583da4720f", "cb73230e998c", "eb48b40fc410", "5ed63f6dd1d0", "74e369bfba70", "013e6e0913db", "9f6c5f89f71d", "377cdc4db798", "fe0d803e8e69", "36c15019e1df", "f2b7fc7df0b8", "c1a29c8ff9f2", "e8c7299f1f53", "0b7978dfe5a8", "a936587f2f4c", "60454894ad64", "25695d831d50", "f96699275b1f", "f1b2a2e5129c", "06dbc15fe872", "8f5a5a482f44", "b479705f60ba", "3e57372f2036", "834629b37d7a", "a581bc478b60", "98a89d9f7c89", "558f642654c1", "e4ec6339a619", "89d8382c8c70", "7b11dd3b38bd", "fe8e8f82d661", "749c3b45e5ca", "2cf709364372", "ce2b908ca32f", "70fb41193e00", "c36b6a681198", "e862791bfb8e", "81c32f856f7a", "81be69ca28ba", "757223e94092", "fa5e2152d235", "ec55d3ee2d4c", "dd28b45f4f42", "50b4ac778e06", "2e68586089e3", "59e642634c96", "eb0e02dfd8b4", "94e0d26f650e", "846e9e1fdcdd", "d63410e53db0", "4fd324a25fa0", "4d9a6d4c5508", "4e010c857a3f", "d28e0f537a2f", "bbd4ca5ff0b4", "456421318c27", "01b5eec535ac", "fde55fdc5614", "db4b951900b9", "dc3dde570034", "62aea50b877d", "3ec6ca357336", "481d2dc9426e", "2de9bd1e2df6", "3c788a70c26f", "0c3c5d02e007", "b10db01c8dba", "518588f30125", "f2db933856b5", "d2247b3677d8", "e574ec9f01d6", "1c0664274886", "d88d9564b697", "6cb3d0159abd", "9c8a0fd5a475", "bae2b1b029a2", "79e755a4676a", "bcdd3f39a7cd", "28fca23bde64", "3edfd122fe96", "9c078a6e6d2a", "a8f3a4e07c6e", "471e941fc5a8", "acbd49406de1", "165c1db3a617", "b32960a7bd10", "009319596f10", "5af6c7fa6105", "2902563da3cd", "1a9de7d076f1", "62bbcbb0ff71", "010b4f492abc", "cafc5d03497e", "4842d4b67551", "28caf9cf9b62", "366a844b4ef9", "703a0ad36217", "21294b636211", "28e35db152b1", "1bf369e7caec", "684d6e83ff67", "8903ee13395b", "c9fed203f285", "9019817d4296", "a7cf6040f8c3", "b63e7a255355", "c6ae15ec89da", "28b2994b14e6", "3209e8bb4565", "52e088b609ce", "5870e437791e", "da009b83a5a9", "10109cb1303f", "fd28a2f58969", "e3220ae6d465", "9f34b95642d1", "94e78eea96d7", "ccbeb5cd7e08", "8319c348413f", "b9f102f3629d", "90916bb3075c", "a99148d94905", "6f08434ca6fe", "ca2ea8d8df2b", "2b1f4c4193ea", "1448456c60ab", "1bacfe585338", "73138fa4e741", "675cb9064a4e", "cc67b4b173e1", "1275d7931666", "aa15fb384fda", "18b8d91e2065", "8d0af6ecb436", "e789895ca549", "050bfca264cc", "1974d4e8fbb0", "c26a3cf55fb8", "0149bc156d4e", "2dce4291ca96", "f27e5dcd9e44", "af902e2cc6a9", "41bbfa104c32", "f605dcb7bd34", "685388d5ebc3", "6c8e91540cc2", "ef4d9cb1aa8f", "c439b66fdbee", "8ba643fc7ec9", "f8a6eef2e15e", "1b14313af370", "407448cca817", "50eee5659ac0", "f76e637a8a8b", "e36f3339ebda", "8d9a3f6125dd", "783083f3600d", "eb31dd23a4d1", "42bb004e9715", "32c169f002f6", "79515c8c74de", "e169f328516d", "aa81d9ddc35c", "b954dd40b7e6", "36d7fdc46760", "d1499a2e70be", "a2ca2076246b", "17e06713dd07", "ec8955c4d497", "2016d4dfbf54", "2c988e32c7ff", "17d03ad0b320", "f9487d76af35", "2f36c46f6bf7", "17b207b0ab85", "9a8479f23334", "83c86efe39ca", "7b484256b065", "b123b70e7f13", "e6772c5f40bc", "d4266bd03660", "2b17c5c7c7aa", "fc7147c8fa09", "25204274346e", "4589e7029c6c", "4190cd324211", "e431e1e4eb35", "ff4344ceda23", "0ebadec0ddf4", "309b1a1a97a1", "c753fbf16887", "ae014ec20eaf", "9718cdebc1d7", "6a1bacde6fbc", "88ec3e3a3b39", "d696238980c7", "fbf8b7917265", "d050368bd72e", "bcf088b8948e", "a330809e453a", "f9dfd4739c0a", "b5591dfad328", "8200ec5bbef0", "92bc9093255c", "49bd8c86a240", "95441f16ba27", "b54de780519b", "c79c86e168c7", "228cd7ee6db4", "dae024541015", "183e119211dc", "a99771517e09", "b6c03222317b", "662903a87a5c", "e43ab7f90b13", "918cbf8dd789", "512238d8c4ed", "da0303be8c63", "c740e7f14118", "733575b2e1c0", "05204897132f", "61224147bce9", "5d60729c9863", "325fa3fd97dc", "08a53de5fe91", "feaf3d71245b", "547b79fe1c94", "adcc6cff0da3", "370c6e5c190c", "be0200da0cd1", "7899aebc76ec", "3055846bea2e", "a872f101f030", "7b495fd33fe2", "069db8c53543", "aa240b035f1e", "d7424a0fc711", "151cdebcaf50", "47bddc031e90", "79cc731d5512", "8950e47cb1fc", "15521caa1663", "ad2303a66b17", "fa8bd2783791", "53fdc7d43053", "c1532f2b2b5a", "af58de60bf12", "9bacae2e38c8", "b114ac3e777f", "aa5930e31577", "5128922745fa", "c62c3d10a80a", "19b7c525cc03", "f142d9184101", "e3744b9c3904", "8854f69d6a12", "879247f04dad", "968c3c33d291", "3186ea682123", "b49a71059a1d", "64fff82dca7f", "e096e9958b1f", "9f52c3575b02", "0018b3765972", "58b994285688", "c23d57dc401c", "30407f196ede", "d9301ced09fd", "d66bca0a1674", "5d5994f66013", "ab92e6e427e0", "490ee4d93833", "be4157d15d99", "2071bc2bfd5d", "9e58eb11a830", "56ad78832648", "36adadac8e6a", "b931a4b74657", "f91cdce9b928", "103c4b3cb7fb", "06441208f3b0", "3c7e7c76c372", "d03b6f6ef96e", "a8ad3512bae9", "5bc450bda4f3", "a1723efe6bc5", "5ae8ea8f83ba", "72d38b4657e4", "f9514e572d74", "cb864247516e", "2f5ca36b8e33", "f524aa5e008e", "78ad8efffd2b", "c7a9b64d0f22", "233339181fdb", "4627973c538c", "6eaf55607348", "04e38dc0b5c5", "bc519f453920", "d1d856edb2b4", "23ed532ce2aa", "d2564ddecdb4", "01f1666746f7", "72a7c88f87dd", "1895a8b488e8", "6bb4fc52bdb9", "4b166665fbc7", "a9775872edf6", "f0da7db68cc5", "3237a36223fd", "dd5ba30ce4aa", "58796272402d", "0b9b49be1ce5", "6485547fed86", "0859f25860ff", "2b4c16598e0a", "018b5e011a51", "65989a9b96a4", "b1e77004f9d4", "308542c3e6a8", "1dc8b1aca1dd", "a24609befd4d", "08505d831bae", "2aa9cf50de8d", "2fae619734c1", "e40e3c3ca69b", "96c4a755994b", "50bd8da8d25c", "6631305e0837", "1718c8a762f5", "81b358ff021b", "2c6896c7f789", "ceb988095704", "f35410f5baf2", "b4284117bbfa", "5ed75ada0647", "1b3f50150855", "bbeacc1835dd", "8bef7f443e04", "0c464c878163", "fe2662b018cf", "3f28cad1a466", "ce8ca8bcabaf", "dd02f55e1a64", "25c1367a3b2f", "a4d4dc2f86ee", "4e50547116c4", "9170ec895d00", "c68e54c33676", "8e075faa5daa", "2bef1bed9374", "c58943e3f81a", "594052a105c8", "79e8a6d3ebeb", "8a68f1ba7dd1", "542f1efb3790", "1d40ce600603", "ac6f983c2b77", "6b4d42ec2230", "c97a93be6e85", "7429e7c11629", "efa210449ed9", "849ba04c88c4", "6f82ee0a8d9e", "fe3d8be7a0f3", "4ed6d985e21c", "7e4812f89af7", "a31d8160d0a5", "1ff2fb27afce", "ffeb7ecdc58a", "9baa7258356a", "6a76e327a8ba", "0e0d9e0c7a05", "280f6d90da04", "5b52b6e37fff", "b3fdcc5fbe9c", "17655fa1eb21", "aad431bc2167", "1a80e8f9971c", "85322cd01386", "fd466d4ee50a", "425824586098", "39da82c27168", "ee1d6ad0ec5f", "f3b8d6e26800", "78fed099f1a9", "1f5b2326e114", "bb1edf74fc7a", "51308cc57265", "5570ce2b34f6", "18a182b267ca", "f8168e96c47f", "7bdadbe0b5d0", "a588aeb65488", "7c9c04c6ba95", "0ede7d220d18", "a3252f9e82a2", "370a2c6d6c62", "dcf6a320bf86"]
import time; T0 = time.time()

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd

def sh(cmd, check=True):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out[-6000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')
    return r

def find_segs(pattern):
    seen = {}
    for p in sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True)):
        seen.setdefault(os.path.basename(p), p)
    return [seen[k] for k in sorted(seen)]

sh('nvidia-smi -L; nproc; free -g | head -2; df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null; which nvcc && nvcc --version | tail -2; cmake --version | head -1', check=False)
KIMI = find_segs('kimik3-q1-seg*-of-23-*.gguf')
assert len(KIMI) == 23, f'{len(KIMI)} Kimi-Segmente gefunden, 23 erwartet'
print('Kimi-Segmente:', round(sum(os.path.getsize(p) for p in KIMI) / 2**30), 'GiB')
patches = sorted(glob.glob('/kaggle/input/**/ntl-kimi-k3-segrun.patch', recursive=True))
PATCH = next(p for p in patches if hashlib.sha256(open(p, 'rb').read()).hexdigest() == PATCH_SHA)
TRAIN = sorted(glob.glob('/kaggle/input/**/name-that-llm/train.csv', recursive=True))[0]
train = pd.read_csv(TRAIN).astype({'id': str}).set_index('id')
assert set(PROBE_IDS + REST_IDS) == set(train.index) and len(PROBE_IDS) + len(REST_IDS) == len(train)
OUT = Path('/kaggle/working/out'); OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/kaggle/temp/work'); WORK.mkdir(parents=True, exist_ok=True)
train[['model']].reset_index().to_csv(OUT / 'train_labels.csv', index=False)
print('Patch', PATCH, '| train', TRAIN, len(train))

In [ ]:
# ---- ntl-segrun bauen: llama.cpp b12247 + Kimi-Patch, CUDA sm_75, llama/ggml statisch ----
t = time.time()
SRCDIR = Path('/kaggle/temp/llama.cpp')
if not SRCDIR.exists():
    sh(f'git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp {SRCDIR}')
    sh(f'cd {SRCDIR} && git apply {PATCH} && git status --short | head -20')
if shutil.which('cmake') is None:
    sh('pip install -q cmake ninja')
# Flags wie José (estopadilla/ntl-glm-likelihood-train): GGML_CUDA_NO_VMM=ON, weil Kaggle kein unversioniertes
# libcuda.so hat (sonst fehlt CUDA::cuda_driver, der Generate-Schritt scheitert und erst der Linker bricht ab,
# so geschehen in Version 1 nach 27 min). Alle Treiber-API-Aufrufe in ggml-cuda hängen an GGML_USE_VMM.
sh(f'cd {SRCDIR} && cmake -S . -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_CUDA_NO_VMM=ON '
   f'-DGGML_CUDA_NCCL=OFF -DGGML_NATIVE=OFF -DBUILD_SHARED_LIBS=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_CURL=OFF '
   f'-DLLAMA_BUILD_SERVER=OFF -DCMAKE_BUILD_TYPE=Release > /tmp/cmake.log 2>&1; '
   f'grep -n -A12 "CMake Error" /tmp/cmake.log | head -60; tail -3 /tmp/cmake.log', check=False)
_log = open('/tmp/cmake.log').read()
assert 'CMake Error' not in _log and 'Generate step failed' not in _log, 'cmake fehlgeschlagen (siehe oben)'
sh(f'cd {SRCDIR} && cmake --build build -j$(nproc) --target ntl-segrun > /tmp/build.log 2>&1; '
   f'grep -n -i "error\\|undefined reference" /tmp/build.log | head -30; tail -3 /tmp/build.log', check=False)
built = SRCDIR / 'build' / 'bin' / 'ntl-segrun'
assert built.is_file(), 'Build fehlgeschlagen (siehe /tmp/build.log oben)'
BIN = Path('/kaggle/working/ntl-bin-kimi'); (BIN / 'lib').mkdir(parents=True, exist_ok=True)
shutil.copy(built, BIN / 'ntl-segrun'); (BIN / 'ntl-segrun').chmod(0o755)
# CUDA-Laufzeit neben das Binary (wie José): das Abgabe-Image hat ein älteres Toolkit; libcublasLt lädt libcublas
import re
def _deps(p):
    return dict(re.findall(r'(libcu\w+\.so\.\d+) => (\S+)', subprocess.run(['ldd', str(p)], capture_output=True, text=True).stdout))
_libs = _deps(BIN / 'ntl-segrun')
print('CUDA-Abhängigkeiten:', _libs)
for k, v in list(_libs.items()):
    if k.startswith('libcublas.'):
        _libs.update({n: q for n, q in _deps(v).items() if n.startswith('libcublasLt')})
for name, p in sorted(_libs.items()):
    if name.startswith(('libcudart', 'libcublas')):
        shutil.copy(os.path.realpath(p), BIN / 'lib' / name)
(BIN / 'BUILD.json').write_text(json.dumps({'llama_tag': LLAMA_TAG, 'patch_sha256': PATCH_SHA,
                                            'binary_sha256': hashlib.sha256(open(BIN / 'ntl-segrun', 'rb').read()).hexdigest()}, indent=1))
ENV = dict(os.environ, LD_LIBRARY_PATH=str(BIN / 'lib') + ':' + os.environ.get('LD_LIBRARY_PATH', ''))
sh(f'ls -la {BIN} {BIN}/lib; cat {BIN}/BUILD.json')
print(f'Build {time.time() - t:.0f} s', flush=True)

In [ ]:
# ---- Metadaten und Chat-Template aus Segment 0; Präfix = Template mit leerer Nutzernachricht ----
sys.path.insert(0, str(SRCDIR / 'gguf-py'))
from gguf import GGUFReader
rd = GGUFReader(KIMI[0])
def val(f):
    try:
        return f.contents()
    except Exception:
        return None
meta = {}
for k, f in rd.fields.items():
    if k.startswith('tokenizer.ggml.') and k not in ('tokenizer.ggml.model', 'tokenizer.ggml.pre', 'tokenizer.ggml.bos_token_id',
                                                     'tokenizer.ggml.eos_token_id', 'tokenizer.ggml.add_bos_token'):
        continue
    v = val(f)
    meta[k] = v if not isinstance(v, (list, tuple)) or len(v) <= 8 else f'[{len(v)} values]'
TEMPLATE = meta.get('tokenizer.chat_template') or ''
json.dump({k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) for k, v in meta.items()},
          open(OUT / 'kimi_meta.json', 'w'), indent=1)
for k, v in meta.items():
    if k != 'tokenizer.chat_template':
        print(k, '=', v)
print('---- chat template ----'); print(TEMPLATE[:4000])
PREFIX = None
try:
    from jinja2.sandbox import ImmutableSandboxedEnvironment
    def _raise(msg):
        raise ValueError(msg)
    env = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True)
    env.globals['raise_exception'] = _raise
    PREFIX = env.from_string(TEMPLATE).render(messages=[{'role': 'user', 'content': ''}], add_generation_prompt=True,
                                              enable_thinking=False, thinking=False, tools=None,
                                              bos_token='', eos_token='')
except Exception as e:
    print('Template nicht renderbar:', repr(e))
if not PREFIX or 'assistant' not in PREFIX:
    PREFIX = ('<|im_system|>system<|im_middle|>You are Kimi, an AI assistant created by Moonshot AI.<|im_end|>'
              '<|im_user|>user<|im_middle|><|im_end|><|im_assistant|>assistant<|im_middle|>')
if PREFIX.rstrip().endswith('<think>'):
    PREFIX = PREFIX.rstrip() + '</think>'
print('PREFIX =', repr(PREFIX))
(OUT / 'kimi_prefix.txt').write_text(PREFIX)

In [ ]:
Path('/kaggle/temp/v1').mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /kaggle/temp/v1/segio.py
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /kaggle/temp/v1/features.py
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /kaggle/temp/v1/run_pipeline_kimi.py
"""Score texts under a big model (GLM-5.3 / GLM-5.3-Flash / Kimi-K3) that ships as layer-segment GGUFs, with ntl-segrun.

Ours (Experiment 11): run_pipeline.py of José Padilla (estopadilla, Apache 2.0) with three changes for Kimi-K3
(segments of xerneas, patch ntl-kimi-k3-segrun.patch):
- state width: Kimi-K3 carries banked attention-residual checkpoints, so the width grows with the layer,
  n_embd * (1 + ceil(layer / attn_res.block_size)); the disk plan uses the largest boundary width.
- two state files used alternately (in-place needs equal widths, which Kimi segments do not have).
- the timing records, per segment, the load time up to "context ready" and the total time.

Every text goes through segment k before segment k+1 is loaded, so each weight is read once per chunk.
The residual state between segments lives in one file on local disk, rewritten in place.
If the disk cannot hold the state of all texts, the texts are split into chunks (each chunk reads the
model again).

    python run_pipeline.py --bin BIN_DIR --segs 'GLOB' --csv train.csv --out OUT_DIR --tag flash
        [--work /tmp/ntl] [--prefix STR] [--max-text-tokens 320] [--seq-batch 8] [--state-dtype auto]
        [--limit N] [--deadline-s S] [--pack] [--ubatch-div P] [--budget-s S] [--prefetch [--prefetch-after F]]

--pack sorts the texts by token count (longest first) and packs them into padded decodes (see ntl-segrun.cpp);
the scores are put back in the input order. --budget-s is a hard limit for the whole call: every ntl-segrun call
gets the time left as its timeout, and after each segment the time of the rest is projected from the segments
done so far; if it does not fit, the call stops with exit code 3 (BudgetExceeded) and writes nothing.
--prefetch: once ntl-segrun has loaded segment k ("context ready"), a background thread reads segment k+1 once
(to /dev/null), so its pages are in the page cache when it is mapped; during the last segment it reads
--prefetch-after (e.g. the first segment of the next model). The same bytes, only earlier: scores are unchanged.

Writes OUT_DIR/{tag}_scores.npz (per-token logprob/entropy/m2/rank, offsets, ids), {tag}_features.csv
and {tag}_timing.json.
"""
import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t in (4, 5), (key, t)   # ours: int32 accepted too
                return struct.unpack('<I' if t == 4 else '<i', f.read(4))[0]
            skip(t)
    return None


class BudgetExceeded(RuntimeError):
    """The time budget (--budget-s) is not enough: stop and let the caller fall back."""


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


LAST_READY_S = [None]   # ours: seconds from start to "context ready" of the last run() call (load time)


def run(cmd, log, timeout=None, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    LAST_READY_S[0] = None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    expired = threading.Event()

    def kill():
        expired.set()
        p.kill()
    timer = threading.Timer(timeout, kill) if timeout is not None else None
    if timer is not None:
        timer.daemon = True
        timer.start()
    lines = []
    for line in p.stdout:
        lines.append(line)
        if 'context ready' in line and LAST_READY_S[0] is None:
            LAST_READY_S[0] = time.time() - t0
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    if timer is not None:
        timer.cancel()
    out = ''.join(lines)
    if expired.is_set():
        with open(log, 'a') as lf:
            lf.write('$ ' + ' '.join(cmd) + '\n' + out + '\n[killed: timeout]\n')
        raise BudgetExceeded(f'timeout after {timeout:.0f} s: {" ".join(cmd)}')
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='glob of the segment files')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--deadline-s', type=float, default=0, help='stop before starting a chunk after this many seconds')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--budget-s', type=float, default=0, help='hard time limit of the whole call (0 = none)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    a = ap.parse_args()
    try:
        main_run(a)
    except BudgetExceeded as e:
        print(f'[{a.tag}] BUDGET EXCEEDED: {e}', flush=True)
        sys.exit(3)


def main_run(a):

    t_start = time.time()

    def left():
        return None if not a.budget_s else a.budget_s - (time.time() - t_start)

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    # ours: --segs may also be a JSON list (segments spread over several datasets)
    segs = sorted(json.loads(a.segs) if a.segs.lstrip().startswith('[') else glob.glob(a.segs),
                  key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
         '--max-text-tokens', str(a.max_text_tokens)], log)
    seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    kimi_embd = gguf_kv_u32(segs[0], 'kimi-k3.embedding_length')
    n_files = 1
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    elif kimi_embd is not None:
        # ours: [inpL | banked checkpoints] per token, n_embd * (1 + ceil(l / bs)) at layer boundary l
        bs = gguf_kv_u32(segs[0], 'kimi-k3.attn_res.block_size') or 0
        n_layer = gguf_kv_u32(segs[0], 'kimi-k3.block_count')
        bounds = [e for _s, e in ranges if e is not None and (n_layer is None or e < n_layer)]
        widths = [kimi_embd * (1 + ((l + bs - 1) // bs if bs > 0 else 0)) for l in bounds] or [kimi_embd]
        width = max(widths)
        n_files = 2   # state in and state out live side by side (widths differ between segments)
        print(f'[{a.tag}] kimi-k3: n_embd {kimi_embd}, attn_res block {bs}, n_layer {n_layer}, '
              f'boundary widths {widths}', flush=True)
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 * n_files <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el * n_files))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div, 'budget_s': a.budget_s}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    st_alt = [st, os.path.join(a.work, f'{a.tag}_state_b.bin')] if n_files == 2 else [st, st]
    done = 0
    for ci, idx in enumerate(chunks):
        if a.deadline_s and time.time() - t_start > a.deadline_s:
            print(f'[{a.tag}] deadline reached before chunk {ci}, stopping', flush=True)
            break
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        seg_load = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st_alt[(k - 1) % 2]]
                cmd += ['--out', sc if k == n_seg - 1 else st_alt[k % 2]]
                if left() is not None and left() <= 0:
                    raise BudgetExceeded('no time left')
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, timeout=left(), on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            seg_load.append(LAST_READY_S[0])
            if k > 0 and n_files == 2 and os.path.exists(st_alt[(k - 1) % 2]):
                os.remove(st_alt[(k - 1) % 2])   # ours: the input state of this segment is no longer needed
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(load {LAST_READY_S[0] or float("nan"):.0f} s, total {time.time()-t_start:.0f} s)', flush=True)
            if a.budget_s:
                # projection: the remaining segments of this chunk at the mean so far (the last one, with the
                # softmax over the vocabulary, counts double), plus the other chunks at this chunk's pace
                rest = n_seg - k - 1
                per = float(np.mean(seg_t))
                proj = per * (rest + (1 if rest > 0 else 0)) + per * n_seg * 1.15 * (len(chunks) - ci - 1)
                if proj > left():
                    raise BudgetExceeded(f'projected {proj:.0f} s for the rest, {left():.0f} s left')
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'n_rows': int(n_rows[idx].sum()), 'seg_seconds': seg_t,
                                 'seg_load_seconds': seg_load})
        for f in (ctk, sc, *st_alt):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
sys.path.insert(0, '/kaggle/temp/v1')
def run_set(tag, ids):
    csv = WORK / f'{tag}.csv'
    train.loc[ids, ['text']].reset_index().to_csv(csv, index=False)
    cmd = [sys.executable, '/kaggle/temp/v1/run_pipeline_kimi.py', '--bin', str(BIN), '--segs', json.dumps(KIMI),
           '--csv', str(csv), '--out', str(WORK / tag), '--tag', 'kimi', '--work', str(WORK / 'state'),
           '--seq-batch', str(SEQ_BATCH), '--state-dtype', 'auto', '--max-text-tokens', str(MAX_TEXT_TOKENS),
           '--prefix', PREFIX, '--prefetch']
    t = time.time()
    with open(OUT / f'{tag}_pipeline.txt', 'w') as log:
        code = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT, env=ENV)
    print(open(OUT / f'{tag}_pipeline.txt').read()[-3000:], flush=True)
    assert code == 0, f'{tag}: Exit {code}, siehe {tag}_pipeline.txt'
    for f in ('kimi_features.csv', 'kimi_timing.json', 'kimi_scores.npz'):
        shutil.copy(WORK / tag / f, OUT / f'{tag}_{f}')
    shutil.rmtree(WORK / tag, ignore_errors=True)
    return time.time() - t, json.load(open(OUT / f'{tag}_kimi_timing.json'))

def auc_report(tag_list):
    from sklearn.metrics import roc_auc_score
    f = pd.concat([pd.read_csv(OUT / f'{t}_kimi_features.csv', dtype={'id': str}) for t in tag_list]).set_index('id')
    y = (train.loc[f.index, 'model'] == 'kimi-k3').to_numpy()
    rows = [(c, roc_auc_score(y, f[c])) for c in f.columns if f[c].std() > 0]
    rows.sort(key=lambda r: -abs(r[1] - 0.5))
    print(f'{len(f)} Texte ({y.sum()} Kimi): AUC Kimi gegen den Rest je Merkmal')
    for c, a in rows[:10]:
        print(f'  {c:16s} {a:.3f}')
    g = f.join(train['model']).groupby('model')[['lp_mean', 'z_mean', 'top1', 'bpc']].mean().round(3)
    print(g.sort_values('lp_mean', ascending=False).to_string())

# ---- Probe ----
dt, tm = run_set('probe', PROBE_IDS)
ch = tm['chunks']
load = sum(sum(x or 0 for x in c['seg_load_seconds']) for c in ch)
total = sum(sum(c['seg_seconds']) for c in ch)
rows = sum(c['n_rows'] for c in ch)
row_s = max(total - load, 0) / rows
PROBE = {'n_texts': tm['n_texts'], 'rows': rows, 'passes': len(ch), 'load_s': load, 'total_s': total, 'row_s': row_s,
         'load_per_pass_s': load / len(ch), 'wall_s': dt}
json.dump(PROBE, open(OUT / 'probe_summary.json', 'w'), indent=1)
print(json.dumps(PROBE, indent=1))
for n_texts in (200, 400, 800):   # Kimi-Stufe im Bewertungslauf: ein Durchlauf + Rechenzeit (160 Token/Text)
    print(f'  Schätzung Testlauf {n_texts} Texte: {(PROBE["load_per_pass_s"] + row_s * n_texts * 165) / 60:.0f} min')
auc_report(['probe'])
GO_ON = PROBE['load_per_pass_s'] <= GATE_LOAD_S and row_s <= GATE_ROW_S
print('GATE:', 'weiter mit allen train-Texten' if GO_ON else 'STOPP: Kimi passt nicht ins Zeitlimit', flush=True)

In [ ]:
# ---- übrige train-Texte, so viele wie bis BUDGET_H passen ----
done_tags = ['probe']
if GO_ON:
    rem = list(REST_IDS)
    tpc = PROBE['rows'] / train.loc[PROBE_IDS, 'text'].str.len().sum()
    ci = 0
    while rem:
        left = BUDGET_H * 3600 - (time.time() - T0)
        # Zeilen grob aus den Zeichen, Kosten = ein Durchlauf + Rechenzeit, 20 % Puffer
        rows_i = np.minimum(train.loc[rem, 'text'].str.len().to_numpy() * tpc, MAX_TEXT_TOKENS) + 20
        cost = 1.2 * (PROBE['load_per_pass_s'] + PROBE['row_s'] * np.cumsum(rows_i))
        n = int(min(CHUNK_MAX, (cost <= left).sum()))
        if n < 50:
            print(f'Stopp: {left/3600:.2f} h übrig, reicht nicht für 50 weitere Texte', flush=True)
            break
        tag = f'c{ci}'
        dt, tm = run_set(tag, rem[:n])
        rem = rem[n:]
        done_tags.append(tag)
        json.dump({'done_tags': done_tags, 'remaining': len(rem), 'hours': (time.time() - T0) / 3600},
                  open(OUT / 'progress.json', 'w'), indent=1)
        print(f'{tag}: {n} Texte in {dt/60:.0f} min, noch {len(rem)}, gesamt {(time.time()-T0)/3600:.2f} h', flush=True)
        ci += 1
feats = pd.concat([pd.read_csv(OUT / f'{t}_kimi_features.csv', dtype={'id': str}) for t in done_tags])
feats = feats.set_index('id').join(train['model'])
feats.reset_index().to_parquet(OUT / 'kimi_train_feats_L160.parquet', index=False)
print(feats.shape, f'gesamt {(time.time()-T0)/3600:.2f} h'); print(feats['model'].value_counts().to_string())
auc_report(done_tags)
shutil.rmtree(WORK, ignore_errors=True)